# Data Preparation — Station Bounding Box Filter

Filters the full Australia-wide station metadata down to stations within a bounding box around the Sydney region, then saves the result as the pipeline's `station_metadata.csv`.

In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster
from pathlib import Path

/Users/nir/Documents/uni/capstone/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Load full Australia-wide station metadata

No header row — columns are: `name, station_id, data_owner, latitude, longitude`

In [2]:
ALL_STATIONS_PATH = Path("all_stations.csv")

all_stations_df = pd.read_csv(
    ALL_STATIONS_PATH,
    header=None,
    names=["name", "station_id", "data_owner", "latitude", "longitude"],
)

# Cast coordinates to float (some rows may have quoted strings)
all_stations_df["latitude"]  = pd.to_numeric(all_stations_df["latitude"],  errors="coerce")
all_stations_df["longitude"] = pd.to_numeric(all_stations_df["longitude"], errors="coerce")

# Drop rows with missing coordinates
all_stations_df = all_stations_df.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

print(f"Total stations loaded: {len(all_stations_df):,}")
print(f"\nLatitude  range: {all_stations_df['latitude'].min():.4f}  →  {all_stations_df['latitude'].max():.4f}")
print(f"Longitude range: {all_stations_df['longitude'].min():.4f}  →  {all_stations_df['longitude'].max():.4f}")
all_stations_df.head()

Total stations loaded: 8,169

Latitude  range: -43.3341  →  -11.1489
Longitude range: 113.7580  →  153.5084


,name,station_id,data_owner,latitude,longitude
0,1 Cooya Pooya,504019,WA - Department of Water and Environmental Reg...,-21.037818,117.117457
1,"10 Mile Brook Dam Water Level Daily Value, Sou...",PI_357693.1,WA - Water Corporation,-33.963617,115.124899
2,10A,GW040994,NSW - Water NSW,-34.483856,150.566715
3,11A,GW040997,NSW - Water NSW,-34.526630,150.564622
4,12/04,70811674,WA - Department of Water and Environmental Reg...,-21.577449,117.073423


## 2. Define the Sydney bounding box

Coordinates are WGS84 decimal degrees.

| Corner | Lat | Lon |
|---|---|---|
| North | −33.20 | — |
| South | −34.80 | — |
| West  | — | 149.90 |
| East  | — | 151.60 |

Adjust the values below to tighten or widen the region.

In [3]:
# ── Bounding box definition ──────────────────────────────────────────────────
SYDNEY_BBOX = {
    "lat_min": -34.80,   # southern boundary
    "lat_max": -33.20,   # northern boundary
    "lon_min":  149.90,  # western boundary
    "lon_max":  151.60,  # eastern boundary
}

print("Sydney bounding box:")
for k, v in SYDNEY_BBOX.items():
    print(f"  {k}: {v}")

Sydney bounding box:
  lat_min: -34.8
  lat_max: -33.2
  lon_min: 149.9
  lon_max: 151.6


## 3. Apply the bounding box filter

In [4]:
lat_mask = (
    (all_stations_df["latitude"]  >= SYDNEY_BBOX["lat_min"]) &
    (all_stations_df["latitude"]  <= SYDNEY_BBOX["lat_max"])
)
lon_mask = (
    (all_stations_df["longitude"] >= SYDNEY_BBOX["lon_min"]) &
    (all_stations_df["longitude"] <= SYDNEY_BBOX["lon_max"])
)

sydney_df = all_stations_df[lat_mask & lon_mask].copy().reset_index(drop=True)

print(f"Stations inside Sydney bbox : {len(sydney_df):,}")
print(f"Stations outside            : {len(all_stations_df) - len(sydney_df):,}")
print(f"\nLatitude  range (filtered): {sydney_df['latitude'].min():.4f}  →  {sydney_df['latitude'].max():.4f}")
print(f"Longitude range (filtered): {sydney_df['longitude'].min():.4f}  →  {sydney_df['longitude'].max():.4f}")
sydney_df.head(10)

Stations inside Sydney bbox : 425
Stations outside            : 7,744

Latitude  range (filtered): -34.7778  →  -33.2064
Longitude range (filtered): 149.9156  →  151.5625


,name,station_id,data_owner,latitude,longitude
0,10A,GW040994,NSW - Water NSW,-34.483856,150.566715
1,11A,GW040997,NSW - Water NSW,-34.526630,150.564622
2,1A,GW040955,NSW - Water NSW,-34.512654,150.524219
3,1M,GW040996,NSW - Water NSW,-34.511769,150.523349
4,1M3d,GW075176,NSW - Water NSW,-34.517254,150.523986
5,1M3s,GW075175,NSW - Water NSW,-34.517262,150.523999
6,2AD1p,GW075216,NSW - Water NSW,-34.540115,150.598953
7,2B,GW040971,NSW - Water NSW,-34.524680,150.587614
8,2C,GW040982,NSW - Water NSW,-34.524189,150.588900
9,2D,GW041039,NSW - Water NSW,-34.522865,150.566863


## 4. Breakdown by data owner

In [5]:
owner_counts = sydney_df["data_owner"].value_counts()
print(f"Unique data owners: {len(owner_counts)}\n")
print(owner_counts.to_string())

Unique data owners: 4

data_owner
NSW - Water NSW                                  305
NSW - Sydney Water Corporation (Sydney Water)    109
NSW - Central Coast Council                        6
NSW - Gosford City Council                         5


## 5. Plot stations on a real map (folium)

In [6]:
# ── Assign a distinct colour per data owner (top owners; rest → grey) ────────
top_owners = sydney_df["data_owner"].value_counts().head(8).index.tolist()

PALETTE = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3",
    "#ff7f00", "#a65628", "#f781bf", "#999999",
]
owner_colour = {owner: PALETTE[i] for i, owner in enumerate(top_owners)}

def get_colour(owner):
    return owner_colour.get(owner, "#cccccc")

# ── Build the folium map centred on Sydney ───────────────────────────────────
centre_lat = (SYDNEY_BBOX["lat_min"] + SYDNEY_BBOX["lat_max"]) / 2
centre_lon = (SYDNEY_BBOX["lon_min"] + SYDNEY_BBOX["lon_max"]) / 2

m = folium.Map(
    location=[centre_lat, centre_lon],
    zoom_start=9,
    tiles="CartoDB positron",   # clean light basemap; swap to "OpenStreetMap" for more detail
)

# ── Bounding box rectangle ───────────────────────────────────────────────────
folium.Rectangle(
    bounds=[
        [SYDNEY_BBOX["lat_min"], SYDNEY_BBOX["lon_min"]],
        [SYDNEY_BBOX["lat_max"], SYDNEY_BBOX["lon_max"]],
    ],
    color="black",
    weight=1.5,
    dash_array="6 4",
    fill=False,
    tooltip="Sydney bounding box",
).add_to(m)

# ── One FeatureGroup per data owner (toggleable in the layer control) ─────────
groups = {}
for owner in top_owners + ["Other"]:
    groups[owner] = folium.FeatureGroup(name=owner, show=True)

for _, row in sydney_df.iterrows():
    owner = row["data_owner"]
    group_key = owner if owner in top_owners else "Other"
    colour = get_colour(owner)

    popup_html = f"""
        <b>{row['name']}</b><br>
        Station ID : {row['station_id']}<br>
        Owner      : {owner}<br>
        Lat / Lon  : {row['latitude']:.5f}, {row['longitude']:.5f}
    """

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=colour,
        fill=True,
        fill_color=colour,
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=row["name"],
    ).add_to(groups[group_key])

for g in groups.values():
    g.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# ── Legend ───────────────────────────────────────────────────────────────────
legend_items = "".join(
    f'<li><span style="background:{get_colour(o)};width:12px;height:12px;'
    f'display:inline-block;border-radius:50%;margin-right:6px;"></span>{o}</li>'
    for o in top_owners
)
legend_html = f"""
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;
            padding:10px 14px;border:1px solid #aaa;border-radius:6px;
            font-size:12px;line-height:1.8;max-width:320px;">
  <b>Data owner</b><br>
  <ul style="list-style:none;padding:0;margin:4px 0 0 0;">
    {legend_items}
    <li><span style="background:#cccccc;width:12px;height:12px;
        display:inline-block;border-radius:50%;margin-right:6px;"></span>Other</li>
  </ul>
  <hr style="margin:6px 0">
  Total stations: <b>{len(sydney_df):,}</b>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Save and display
MAP_PATH = "sydney_stations_map.html"
m.save(MAP_PATH)
print(f"Interactive map saved to {MAP_PATH}")
m   # renders inline in Jupyter

Interactive map saved to sydney_stations_map.html


## 7. Save filtered metadata

Saves to `database/australia/station_metadata.csv` — the path the pipeline reads from.  
Format matches the existing file: **no header**, columns = `name, station_id, data_owner, latitude, longitude`.

In [ ]:
OUTPUT_PATH = Path("database/australia/station_metadata.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

sydney_df[["name", "station_id", "data_owner", "latitude", "longitude"]].to_csv(
    OUTPUT_PATH,
    index=False,
    header=False,   # keep headerless to match existing format
)

print(f"Saved {len(sydney_df):,} stations to {OUTPUT_PATH}")
print("\nFirst 5 rows of saved file:")
pd.read_csv(OUTPUT_PATH, header=None,
            names=["name", "station_id", "data_owner", "latitude", "longitude"]).head()

## 8. Sanity check — cross-reference with rainfall data

How many of the filtered stations actually have rainfall records in `all_stations_rainfall_hourly_combined.csv`?

In [ ]:
RAINFALL_CSV = Path("all_stations_rainfall_hourly_combined.csv")

if RAINFALL_CSV.exists():
    # Only read the station_id column to avoid loading the full file
    rainfall_ids = pd.read_csv(RAINFALL_CSV, usecols=["station_id"])["station_id"].unique()

    # station_id in the metadata may be strings; try numeric match
    def to_numeric_safe(val):
        try:
            return int(float(str(val)))
        except (ValueError, TypeError):
            return None

    sydney_df["station_id_int"] = sydney_df["station_id"].apply(to_numeric_safe)
    rainfall_id_set = set(rainfall_ids)

    matched = sydney_df[sydney_df["station_id_int"].isin(rainfall_id_set)]
    unmatched = sydney_df[~sydney_df["station_id_int"].isin(rainfall_id_set)]

    print(f"Stations in bbox          : {len(sydney_df):,}")
    print(f"  ✓ With rainfall data    : {len(matched):,}")
    print(f"  ✗ No rainfall data      : {len(unmatched):,}")
    print("\nStations WITH rainfall data:")
    display(matched[["name", "station_id", "data_owner", "latitude", "longitude"]].reset_index(drop=True))
else:
    print(f"{RAINFALL_CSV} not found — skipping cross-reference.")